In [1]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
translation = pd.read_csv("../data/product_category_name_translation.csv")

# Convert dates
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"],
    errors="coerce"
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"],
    errors="coerce"
)

# Delivery delay
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders["late_delivery_flag"] = (
    orders["delivery_delay_days"] > 0
).astype(int)

# Add English category names
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products["product_category_final"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
)

# Combine order items with product category
items_category = order_items.merge(
    products[["product_id", "product_category_final"]],
    on="product_id",
    how="left"
)

# Add order-level delivery information
items_category = items_category.merge(
    orders[
        [
            "order_id",
            "delivery_delay_days",
            "late_delivery_flag"
        ]
    ],
    on="order_id",
    how="left"
)

# Add review information
items_category = items_category.merge(
    reviews[
        [
            "order_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="left"
)

# Negative review flag
items_category["negative_review_flag"] = (
    items_category["review_score"] <= 3
).astype(int)

# Category-level analysis
category_analysis = (
    items_category
    .groupby("product_category_final")
    .agg(
        revenue=("price", "sum"),
        orders=("order_id", "nunique"),
        average_rating=("review_score", "mean"),
        negative_review_rate=("negative_review_flag", "mean"),
        late_delivery_rate=("late_delivery_flag", "mean"),
        average_delivery_delay=("delivery_delay_days", "mean"),
        total_freight=("freight_value", "sum"),
        total_price=("price", "sum")
    )
    .reset_index()
)

# Convert rates to percentages
category_analysis["negative_review_rate"] *= 100
category_analysis["late_delivery_rate"] *= 100

# Freight ratio
category_analysis["freight_ratio"] = np.where(
    category_analysis["total_price"] > 0,
    category_analysis["total_freight"]
    / category_analysis["total_price"] * 100,
    np.nan
)

# Round values
category_analysis = category_analysis.round({
    "revenue": 2,
    "average_rating": 2,
    "negative_review_rate": 2,
    "late_delivery_rate": 2,
    "average_delivery_delay": 2,
    "freight_ratio": 2
})

print("Number of categories:", len(category_analysis))

print("\nCategory analysis:")
print(
    category_analysis.sort_values(
        "revenue",
        ascending=False
    ).head(15)
)

Number of categories: 73

Category analysis:
   product_category_final     revenue  orders  average_rating  \
43          health_beauty  1263138.54    8836            4.14   
72          watches_gifts  1206075.33    5624            4.02   
7          bed_bath_table  1050936.61    9417            3.90   
67         sports_leisure   993656.51    7720            4.11   
15  computers_accessories   919640.54    6689            3.93   
39        furniture_decor   736282.47    6449            3.90   
20             cool_stuff   637258.51    3632            4.15   
49             housewares   634542.60    5884            4.06   
5                    auto   594363.10    3897            4.07   
42           garden_tools   486432.45    3518            4.04   
71                   toys   484769.90    3886            4.16   
6                    baby   412117.47    2885            4.01   
60              perfumery   400212.94    3162            4.16   
70              telephony   323839.40    4199

In [2]:
# Use median values as relative thresholds
revenue_threshold = category_analysis["revenue"].median()
rating_threshold = category_analysis["average_rating"].median()

print("Revenue threshold:", round(revenue_threshold, 2))
print("Rating threshold:", round(rating_threshold, 2))

# Identify high-revenue + low-satisfaction categories
high_revenue_low_satisfaction = category_analysis[
    (category_analysis["revenue"] >= revenue_threshold) &
    (category_analysis["average_rating"] < rating_threshold)
].copy()

# Sort by revenue so the highest-impact categories appear first
high_revenue_low_satisfaction = (
    high_revenue_low_satisfaction
    .sort_values("revenue", ascending=False)
    .reset_index(drop=True)
)

print("\nHIGH REVENUE + LOW SATISFACTION CATEGORIES")
print("-------------------------------------------")

print(
    high_revenue_low_satisfaction[
        [
            "product_category_final",
            "revenue",
            "orders",
            "average_rating",
            "negative_review_rate",
            "late_delivery_rate",
            "average_delivery_delay",
            "freight_ratio"
        ]
    ].round(2)
)

Revenue threshold: 46457.37
Rating threshold: 4.05

HIGH REVENUE + LOW SATISFACTION CATEGORIES
-------------------------------------------
                     product_category_final     revenue  orders  \
0                             watches_gifts  1206075.33    5624   
1                            bed_bath_table  1050936.61    9417   
2                     computers_accessories   919640.54    6689   
3                           furniture_decor   736282.47    6449   
4                              garden_tools   486432.45    3518   
5                                      baby   412117.47    2885   
6                                 telephony   323839.40    4199   
7                          office_furniture   275224.49    1273   
8                               electronics   160376.64    2550   
9                            consoles_games   158000.22    1062   
10                        home_construction    83207.97     490   
11               agro_industry_and_commerce    72530.47  

In [3]:
# Create a simple business-priority score
# Higher revenue + lower satisfaction = higher priority

category_analysis["satisfaction_gap"] = (
    5 - category_analysis["average_rating"]
)

category_analysis["priority_score"] = (
    category_analysis["revenue"] *
    category_analysis["satisfaction_gap"]
)

priority_categories = (
    category_analysis
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)

print("TOP CATEGORY PRIORITIES")
print("-----------------------")

print(
    priority_categories[
        [
            "product_category_final",
            "revenue",
            "orders",
            "average_rating",
            "negative_review_rate",
            "late_delivery_rate",
            "freight_ratio",
            "priority_score"
        ]
    ].head(15).round(2)
)

TOP CATEGORY PRIORITIES
-----------------------
   product_category_final     revenue  orders  average_rating  \
0           watches_gifts  1206075.33    5624            4.02   
1          bed_bath_table  1050936.61    9417            3.90   
2           health_beauty  1263138.54    8836            4.14   
3   computers_accessories   919640.54    6689            3.93   
4          sports_leisure   993656.51    7720            4.11   
5         furniture_decor   736282.47    6449            3.90   
6              housewares   634542.60    5884            4.06   
7                    auto   594363.10    3897            4.07   
8              cool_stuff   637258.51    3632            4.15   
9            garden_tools   486432.45    3518            4.04   
10       office_furniture   275224.49    1273            3.49   
11                   baby   412117.47    2885            4.01   
12                   toys   484769.90    3886            4.16   
13              telephony   323839.40    4

In [4]:
category_recommendations = {
    "watches_gifts":
        "Prioritize customer-experience improvements because of high revenue and below-median satisfaction.",

    "bed_bath_table":
        "Prioritize improvements in product experience and delivery because of high revenue and low ratings.",

    "computers_accessories":
        "Investigate product quality, seller performance, and delivery issues affecting customer satisfaction.",

    "furniture_decor":
        "Review product quality, damage, freight costs, and delivery performance.",

    "office_furniture":
        "Investigate urgently due to the lowest satisfaction and highest negative-review rate among major categories.",

    "telephony":
        "Investigate product and seller-related complaints while monitoring delivery performance.",

    "home_confort":
        "Review customer complaints and delivery issues because of the high negative-review rate."
}

print("CATEGORY BUSINESS RECOMMENDATIONS")
print("---------------------------------")

for category, recommendation in category_recommendations.items():
    print("\n" + category)
    print("-" * len(category))
    print(recommendation)

CATEGORY BUSINESS RECOMMENDATIONS
---------------------------------

watches_gifts
-------------
Prioritize customer-experience improvements because of high revenue and below-median satisfaction.

bed_bath_table
--------------
Prioritize improvements in product experience and delivery because of high revenue and low ratings.

computers_accessories
---------------------
Investigate product quality, seller performance, and delivery issues affecting customer satisfaction.

furniture_decor
---------------
Review product quality, damage, freight costs, and delivery performance.

office_furniture
----------------
Investigate urgently due to the lowest satisfaction and highest negative-review rate among major categories.

telephony
---------
Investigate product and seller-related complaints while monitoring delivery performance.

home_confort
------------
Review customer complaints and delivery issues because of the high negative-review rate.
